In [2]:
# ========== 单元格 1：读数据 + 重建 df_main ==========
import numpy as np
import pandas as pd

path = r'C:\Users\13676\Desktop\senna_cohort.csv'
df = pd.read_csv(path, encoding='gb18030',
                 encoding_errors='replace', low_memory=False)

time_cols = ['index_time', 'baseline_k_charttime', 'hypokalemia_time',
             'window_event_time', 'admittime', 'dischtime', 'deathtime',
             'baseline_egfr_charttime']
for c in time_cols:
    if c in df.columns:
        df[c] = pd.to_datetime(df[c], dayfirst=True, errors='coerce')

df_main = df.dropna(subset=['senna_exposed', 'hypokalemia_event',
                             'baseline_k', 'age']).copy()
df_main['y'] = df_main['hypokalemia_event'].astype(int)
df_main['senna_exposed'] = df_main['senna_exposed'].astype(int)
df_main['gender_male'] = (df_main['gender'].str.upper().str[0] == 'M').astype(int)

print('df_main 重建完成:', len(df_main), '行')

df_main 重建完成: 41529 行


In [3]:
# ========== 单元格 1：定义亚组 ==========
import numpy as np
import pandas as pd

df_sub = df_main.copy()

# 年龄组（已有 age_group）
# 基线血钾三分层
df_sub['k_group'] = pd.cut(
    df_sub['baseline_k'],
    bins=[3.5, 4.0, 4.5, np.inf],
    labels=['Low (3.5–4.0)', 'Medium (4.0–4.5)', 'High (>4.5)'],
    include_lowest=True)

# 非泻药合并用药数按中位数分割
med_drug = df_sub['drug_count_no_laxatives'].median()
df_sub['drug_group'] = (df_sub['drug_count_no_laxatives'] >= med_drug).astype(int)
print(f'合并用药数中位数：{med_drug}')

# 二分类亚组
# age_group（65-74 / 75-84 / 85+）—— 多分类
# diuretic / chf / ckd —— 已为二分类
print('亚组变量定义完成')
print(df_sub['k_group'].value_counts())

合并用药数中位数：11.0
亚组变量定义完成
k_group
Low (3.5–4.0)       16858
Medium (4.0–4.5)    15939
High (>4.5)          8732
Name: count, dtype: int64


In [4]:
# ========== 单元格 2：亚组分析 ==========
import statsmodels.formula.api as smf

# 主模型协变量（亚组内统一使用，避免亚组样本量太小）
COVARS = ('gender_male + C(age_group, Treatment("65-74")) + '
          'baseline_k + drug_count_no_laxatives + '
          'chf + ckd + dm + diuretic + acei_arb + digoxin + '
          'insulin + statin + anticoagulant + '
          'lactulose + peg + bisacodyl + ppi + beta2_agonist + corticosteroid')

def run_subgroup(df, subgroup_col, level, exclude_cols=()):
    """在指定亚组内跑多因素回归，提取 senna 的 aOR"""
    sub = df[df[subgroup_col] == level].copy()
    if len(sub) < 200:
        return None
    # 如果亚组变量本身在主模型协变量里，需移除避免共线
    covars_use = COVARS
    if 'age_group' in subgroup_col:
        covars_use = covars_use.replace(
            'C(age_group, Treatment("65-74")) + ', '')
    if subgroup_col in ['diuretic', 'chf', 'ckd', 'dm']:
        covars_use = covars_use.replace(f'{subgroup_col} + ', '')
    formula = f'y ~ senna_exposed + {covars_use}'
    try:
        m = smf.logit(formula, data=sub).fit(disp=0)
    except Exception as e:
        return None
    or_ = np.exp(m.params['senna_exposed'])
    lo  = np.exp(m.conf_int().loc['senna_exposed', 0])
    hi  = np.exp(m.conf_int().loc['senna_exposed', 1])
    p   = m.pvalues['senna_exposed']
    return {'Subgroup': subgroup_col,
            'Level': str(level),
            'N': len(sub),
            'N events': int(sub['y'].sum()),
            'aOR': round(or_, 3),
            'CI_low': round(lo, 3),
            'CI_high': round(hi, 3),
            'P': p}

# ---------- 逐个亚组跑 ----------
results = []

# 年龄组
for lv in ['65-74', '75-84', '85+']:
    r = run_subgroup(df_sub, 'age_group', lv, exclude_cols=('age_group',))
    if r: results.append(r)

# 基线血钾
for lv in df_sub['k_group'].cat.categories:
    r = run_subgroup(df_sub, 'k_group', lv)
    if r: results.append(r)

# 利尿剂
for lv in [0, 1]:
    r = run_subgroup(df_sub, 'diuretic', lv, exclude_cols=('diuretic',))
    if r: results.append(r)

# CHF
for lv in [0, 1]:
    r = run_subgroup(df_sub, 'chf', lv, exclude_cols=('chf',))
    if r: results.append(r)

# CKD
for lv in [0, 1]:
    r = run_subgroup(df_sub, 'ckd', lv, exclude_cols=('ckd',))
    if r: results.append(r)

# 合并用药数（中位数分割）
for lv in [0, 1]:
    r = run_subgroup(df_sub, 'drug_group', lv)
    if r: results.append(r)

# ---------- 交互检验 ----------
def interaction_p(df, subgroup_col):
    """在总样本中加入 senna × subgroup 交互项，提取交互 P"""
    sub = df.copy()
    if subgroup_col == 'age_group':
        # 用年龄 ≥85 vs 其他 作为二分类交互
        sub['_sub'] = (sub['age_group'] == '85+').astype(int)
    elif subgroup_col == 'k_group':
        sub['_sub'] = (sub['k_group'] == 'Low (3.5–4.0)').astype(int)
    else:
        sub['_sub'] = sub[subgroup_col].astype(int)
    formula = f'y ~ senna_exposed * _sub + {COVARS}'
    try:
        m = smf.logit(formula, data=sub).fit(disp=0)
        for term in m.pvalues.index:
            if 'senna_exposed:_sub' in term or '_sub:senna_exposed' in term:
                return m.pvalues[term]
    except Exception:
        return np.nan
    return np.nan

interaction_dict = {
    'age_group':   interaction_p(df_sub, 'age_group'),
    'k_group':     interaction_p(df_sub, 'k_group'),
    'diuretic':    interaction_p(df_sub, 'diuretic'),
    'chf':         interaction_p(df_sub, 'chf'),
    'ckd':         interaction_p(df_sub, 'ckd'),
    'drug_group':  interaction_p(df_sub, 'drug_group'),
}

# ---------- 汇总 ----------
table_sub = pd.DataFrame(results)
print(table_sub.to_string(index=False))
print('\n交互作用 P 值：')
for k, v in interaction_dict.items():
    print(f'  {k}: P = {v:.4f}')
table_sub.to_excel('TableS_subgroup_analysis.xlsx', index=False)

  Subgroup            Level     N  N events   aOR  CI_low  CI_high            P
 age_group            65-74 18261      3764 1.197   1.096    1.308 6.887603e-05
 age_group            75-84 14850      3199 1.185   1.079    1.302 3.976727e-04
 age_group              85+  8418      1789 1.158   1.021    1.313 2.255831e-02
   k_group    Low (3.5–4.0) 16858      4945 1.143   1.057    1.237 8.441392e-04
   k_group Medium (4.0–4.5) 15939      2560 1.233   1.112    1.368 6.923216e-05
   k_group      High (>4.5)  8732      1247 1.187   1.017    1.386 2.958706e-02
  diuretic                0 21426      3424 1.003   0.919    1.095 9.433710e-01
  diuretic                1 20103      5328 1.321   1.224    1.426 9.779481e-13
       chf                0 30662      6031 1.125   1.051    1.203 6.607965e-04
       chf                1 10867      2721 1.369   1.228    1.526 1.557110e-08
       ckd                0 31358      6803 1.164   1.091    1.242 4.068859e-06
       ckd                1 10171      1

In [5]:
# ========== 数据准备 ==========
import pandas as pd
import numpy as np

# 将你截图里的数据整理成 DataFrame
raw_data = [
    # Subgroup, Group_Label, Level_Label, N, Events, aOR, CI_low, CI_high, P_int
    ('Age', 'Age group', '65–74', 18261, 3764, 1.197, 1.096, 1.308, 0.346),
    ('Age', 'Age group', '75–84', 14850, 3199, 1.185, 1.079, 1.302, 0.346),
    ('Age', 'Age group', '≥85',   8418, 1789, 1.158, 1.021, 1.313, 0.346),
    
    ('K', 'Baseline potassium', '3.5–4.0', 16858, 4945, 1.143, 1.057, 1.237, 0.003),
    ('K', 'Baseline potassium', '4.0–4.5', 15939, 2560, 1.233, 1.112, 1.368, 0.003),
    ('K', 'Baseline potassium', '>4.5',    8732,  1247, 1.187, 1.017, 1.386, 0.003),
    
    ('Diuretic', 'Diuretic use', 'No',  21426, 3424, 1.003, 0.919, 1.095, 0.0002),
    ('Diuretic', 'Diuretic use', 'Yes', 20103, 5328, 1.321, 1.224, 1.426, 0.0002),
    
    ('CHF', 'Congestive heart failure', 'No',  30662, 6031, 1.125, 1.051, 1.203, 0.0000),
    ('CHF', 'Congestive heart failure', 'Yes', 10867, 2721, 1.369, 1.228, 1.526, 0.0000),
    
    ('CKD', 'Chronic kidney disease', 'No',  31358, 6803, 1.164, 1.091, 1.242, 0.045),
    ('CKD', 'Chronic kidney disease', 'Yes', 10171, 1949, 1.287, 1.137, 1.456, 0.045),
    
    ('Drug', 'Non-laxative drug count', 'Low',  18509, 2596, 1.046, 0.950, 1.152, 0.007),
    ('Drug', 'Non-laxative drug count', 'High', 23020, 6156, 1.264, 1.177, 1.358, 0.007),
]

df_sub = pd.DataFrame(raw_data, columns=[
    'Subgroup', 'Group_EN', 'Level_EN', 'N', 'Events', 'aOR', 'CI_low', 'CI_high', 'P_int'
])

# 中文标签映射（供中文版使用）
df_sub['Group_CN'] = df_sub['Subgroup'].map({
    'Age': '年龄组', 'K': '基线血钾', 'Diuretic': '利尿剂',
    'CHF': '心力衰竭', 'CKD': '慢性肾脏病', 'Drug': '非泻药合并用药数'
})
df_sub['Level_CN'] = df_sub['Level_EN'].map({
    'No': '否', 'Yes': '是', 'Low': '低', 'High': '高'
})
# 血钾和年龄组的中文标签
df_sub.loc[df_sub['Subgroup']=='K', 'Level_CN'] = ['3.5–4.0', '4.0–4.5', '>4.5']
df_sub.loc[df_sub['Subgroup']=='Age', 'Level_CN'] = ['65–74岁', '75–84岁', '≥85岁']

print("数据准备完成")

数据准备完成


In [10]:
# ========== 绘图函数 ==========
import matplotlib as mpl
import matplotlib.pyplot as plt

def plot_subgroup_forest(df, lang='EN', save_path='Figure.png'):
    """
    绘制亚组森林图。
    lang='EN' 使用 Arial 字体；lang='CN' 使用中文字体（SimHei/YaHei）。
    """
    # ---------- 字体设置 ----------
    font_family = 'Arial' if lang == 'EN' else 'SimHei'
    mpl.rcParams['axes.unicode_minus'] = False
    
    # ---------- 构造绘图所需的数据行 ----------
    rows = []
    for group in df['Subgroup'].unique():
        sub = df[df['Subgroup'] == group]
        group_name = sub['Group_EN'].iloc[0] if lang == 'EN' else sub['Group_CN'].iloc[0]
        p_int = sub['P_int'].iloc[0]
        p_int_str = '<0.001' if p_int < 0.001 else f'{p_int:.3f}'
        
        # 添加组标题行（带 P for interaction）
        if lang == 'EN':
            title = f"{group_name} (P for interaction = {p_int_str})"
        else:
            title = f"{group_name}（交互作用 P = {p_int_str}）"
            
        rows.append({'type': 'header', 'label': title})
        
        # 添加子分类行
        for _, r in sub.iterrows():
            lv_name = r['Level_EN'] if lang == 'EN' else r['Level_CN']
            rows.append({
                'type': 'data',
                'label': f"    {lv_name}",
                'n_events': f"{int(r['Events'])}/{int(r['N'])}",
                'aOR': r['aOR'], 'CI_low': r['CI_low'], 'CI_high': r['CI_high'],
                'or_text': f"{r['aOR']:.2f} ({r['CI_low']:.2f}–{r['CI_high']:.2f})"
            })

    n_rows = len(rows)
    
    # ---------- 创建画布：三栏布局 ----------
    fig = plt.figure(figsize=(11, n_rows * 0.38 + 2))
    gs = fig.add_gridspec(1, 4, width_ratios=[3.0, 1.2, 2.5, 1.8], wspace=0.05)
    ax_lab  = fig.add_subplot(gs[0, 0])  # 标签
    ax_n    = fig.add_subplot(gs[0, 1], sharey=ax_lab)  # 事件数
    ax_f    = fig.add_subplot(gs[0, 2], sharey=ax_lab)  # 森林图
    ax_txt  = fig.add_subplot(gs[0, 3], sharey=ax_lab)  # aOR 文本
    
    # 隐藏除森林图外的所有坐标轴
    for ax in [ax_lab, ax_n, ax_txt]:
        ax.axis('off')
    
    y_positions = np.arange(n_rows)[::-1] # 从上到下
    
    # 设置 Y 轴范围
    ax_f.set_ylim(-1, n_rows)
    ax_f.set_xscale('log')
    
       # X 轴范围（对数）
    min_ci = df['CI_low'].min()
    max_ci = df['CI_high'].max()
    xlim = (min_ci * 0.8, max_ci * 1.2)
    ax_f.set_xlim(xlim)
    
    # ===== 修复刻度重叠问题 =====
    import matplotlib.ticker as ticker
    ax_f.xaxis.set_minor_locator(ticker.NullLocator())  # 关闭次要刻度
    # 自定义主刻度位置（根据你的数据范围 0.73 - 1.83 设定）
    ax_f.set_xticks([0.8, 1.0, 1.2, 1.5]) 
    # 将科学计数法格式改为普通小数（如 1.0）
    ax_f.xaxis.set_major_formatter(ticker.FormatStrFormatter('%.1f'))
    # ==========================
    
    # 添加参考线 OR=1
    ax_f.axvline(1, color='gray', linestyle='--', linewidth=1)
    
    # 表头 (y = n_rows)
    header_y = n_rows
    header_kwargs = {'fontsize': 10, 'fontweight': 'bold', 'va': 'center', 'family': font_family}
    ax_lab.text(0.0, header_y, 'Subgroup' if lang == 'EN' else '亚组', ha='left', **header_kwargs)
    ax_n.text(0.5, header_y, 'Events/N' if lang == 'EN' else '事件数/N', ha='center', **header_kwargs)
    ax_txt.text(0.0, header_y, 'aOR (95% CI)', ha='left', **header_kwargs)
    
    # 遍历数据绘制
    for i, row in enumerate(rows):
        y = y_positions[i]
        
        if row['type'] == 'header':
            # 主组标题加粗
            ax_lab.text(0.0, y, row['label'], ha='left', va='center', 
                        fontsize=10, fontweight='bold', family=font_family)
        else:
            # 子分类
            ax_lab.text(0.0, y, row['label'], ha='left', va='center', 
                        fontsize=9, family=font_family)
            ax_n.text(0.5, y, row['n_events'], ha='center', va='center', 
                      fontsize=9, family=font_family)
            ax_txt.text(0.0, y, row['or_text'], ha='left', va='center', 
                        fontsize=9, family=font_family)
            
            # 森林图
            is_highlight = (row['aOR'] > 1.15 and row['CI_low'] > 1.0) # 高亮显著且效应较大的
            color = '#C0392B' if is_highlight else '#2C3E50'
            ms = 7 if is_highlight else 5
            
            ax_f.errorbar(row['aOR'], y, 
                          xerr=[[row['aOR'] - row['CI_low']], [row['CI_high'] - row['aOR']]],
                          fmt='o', color=color, ecolor=color, 
                          capsize=3, elinewidth=1.2, markersize=ms)

    # 设置森林图 X 轴
    ax_f.spines['top'].set_visible(False)
    ax_f.spines['right'].set_visible(False)
    ax_f.spines['left'].set_visible(False)
    ax_f.set_yticks([])
    ax_f.set_xlabel('Adjusted OR' if lang == 'EN' else '调整 OR', fontsize=10, family=font_family)
    ax_f.tick_params(axis='x', labelsize=9)
    for tick in ax_f.get_xticklabels():
        tick.set_family(font_family)
        
    # 标题
    title_str = 'Subgroup Analysis: Senna Exposure and Hypokalemia' if lang == 'EN' else '亚组分析：番泻叶暴露与低钾血症'
    fig.suptitle(title_str, fontsize=13, fontweight='bold', family=font_family, y=0.98)
    
    # 保存
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"已保存：{save_path}")


In [11]:
# ========== 生成中英双版 ==========
# 1. 英文版（Arial）
plot_subgroup_forest(df_sub, lang='EN', save_path='Figure_Subgroup_EN.png')

# 2. 中文版
plot_subgroup_forest(df_sub, lang='CN', save_path='Figure_Subgroup_CN.png')

已保存：Figure_Subgroup_EN.png
已保存：Figure_Subgroup_CN.png
